# 🇵🇭 Philippine Address Parsing & Normalization Pipeline

**Architecture**

```
Raw addresses
     │
     ▼
Stage 1 ── Alias normalization      (ph_address_alias_extended_v4.csv)
     │
     ▼
Stage 2 ── PSGC fuzzy match          (rapidfuzz · province-anchored)
     │
     ▼
Stage 3 ── Confidence gate           (high → fast path · low → API)
     │                    │
     ▼                    ▼
  dim_location        Stage 4 ── Nominatim OSM API verify
  fast lookup              │
     │                    ▼
     └──────── Stage 5 ──────── Output split
                               │               │
                     verified_addresses.xlsx   invalid_addresses.xlsx
```

**Key design rules**
- A street/barangay token that *happens* to share a city name is **not** assigned that city unless a province or region corroborates it (fixes the *South Signal → General Santos* false match)
- Only ambiguous / low-confidence rows hit the Nominatim API — high-confidence rows are resolved purely from `dim_location` for speed
- 10 000 rows target: < 20 min total


## Cell 0 — Environment check
Verify all required packages are present before running the pipeline.

In [68]:
import importlib, sys

REQUIRED = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'rapidfuzz': 'rapidfuzz',
    'tqdm': 'tqdm',
    'openpyxl': 'openpyxl',
    'xlsxwriter': 'xlsxwriter',
}

missing = []
for pkg_label, pkg in REQUIRED.items():
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'n/a')
        print(f'  ✅  {pkg_label:<14} {ver}')
    except ImportError:
        print(f'  ❌  {pkg_label:<14} NOT FOUND')
        missing.append(pkg)

if missing:
    print(f'\n⚠️  Install missing packages:')
    print(f'    pip install {" ".join(missing)}')
else:
    print('\n✅  All dependencies satisfied — ready to run.')


  ✅  pandas         3.0.1
  ✅  numpy          2.4.3
  ✅  rapidfuzz      3.14.3
  ✅  tqdm           4.67.3
  ✅  openpyxl       3.1.5
  ✅  xlsxwriter     3.2.9

✅  All dependencies satisfied — ready to run.


## Cell 1 — Configuration

Edit the paths and thresholds here. Everything else in the pipeline reads from these variables.

| Variable | Default | Meaning |
|---|---|---|
| `CITY_SCORE_HIGH` | 88 | Auto-accept without API call |
| `CITY_SCORE_LOW` | 65 | Minimum to even consider a city match |
| `PROV_SCORE_HIGH` | 88 | Auto-accept province |
| `PROV_SCORE_LOW` | 60 | Minimum province score |
| `USE_API` | `True` | Set `False` for fuzzy-only mode (faster, less accurate) |
| `API_QUERY_MODE` | `'aggressive'` | API search mode: `'strict'` = minimal queries, `'aggressive'` = typo-tolerant fallbacks |
| `MAX_ROWS` | `None` | Set an integer (e.g. `100`) to process only a slice for testing |


In [69]:
from pathlib import Path

# -- File paths --
BASE_DIR = Path("..") / ".." / "data"   # notebook is in update_address/notebooks

INPUT_FILE   = str(BASE_DIR / "sample"  / "sample_org_address.xlsx")
DIM_LOCATION = str(BASE_DIR / "mapping" / "dim_location_20260316_v2.csv")
ALIAS_FILE   = str(BASE_DIR / "utils"   / "ph_address_alias_extended_v4.csv")
OUT_VERIFIED = str(BASE_DIR / "output"  / "verified_addresses.xlsx")
OUT_INVALID  = str(BASE_DIR / "output"  / "invalid_addresses.xlsx")

# -- Batch input (unchanged) --------------------------------------------------
input_paths = [
    Path('../../data/sample/Structured_Philippine_Addresses_Unmatched/') / f'Structured_Philippine_Addresses_Unmatched_part_{i:03d}.xlsx'
    for i in range(1, 51)   # Adjust range to cover your batches, e.g. range(1, 101)
]

# -- Fuzzy-match thresholds (0-100) ------------------------------------------
CITY_SCORE_HIGH  = 88
CITY_SCORE_LOW   = 65
PROV_SCORE_HIGH  = 88
PROV_SCORE_LOW   = 60
BRGY_SCORE_MIN   = 75

# -- API settings kept for optional future use (currently disabled) -----------
PHOTON_URL = 'https://photon.komoot.io/api'
PHOTON_UA  = 'ph-address-pipeline/1.0 (research)'
BATCH_DELAY = 1.1
MAX_RETRIES = 1

# -- Run controls -------------------------------------------------------------
USE_API        = False         # Force fuzzy-only mode for maximum speed
API_QUERY_MODE = 'strict'      # Placeholder when API is re-enabled
MAX_ROWS       = None          # None = all rows; set e.g. 100 for quick test runs

# -- Quick path check ---------------------------------------------------------
if input_paths:
    print(f'  Batch mode enabled: {len(input_paths)} input files')
    for p in input_paths:
        status = 'OK' if p.exists() else 'NOT FOUND'
        print(f'  {status}  {p}')
else:
    status = 'OK' if Path(INPUT_FILE).exists() else 'NOT FOUND'
    print(f'  {status}  {INPUT_FILE}')

for f in [DIM_LOCATION, ALIAS_FILE]:
    status = 'OK' if Path(f).exists() else 'NOT FOUND'
    print(f'  {status}  {f}')
print(f'  USE_API = {USE_API} (fuzzy-only)')


  Batch mode enabled: 50 input files
  OK  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_001.xlsx
  OK  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_002.xlsx
  OK  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_003.xlsx
  OK  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_004.xlsx
  OK  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_005.xlsx
  OK  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_006.xlsx
  OK  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_007.xlsx
  OK  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_p

## Cell 2 — Imports & logging setup

In [70]:
import re
import time
import logging
import urllib.request
import urllib.parse
import json
from typing import Optional

import pandas as pd
import numpy as np
from IPython.display import display
from rapidfuzz import fuzz, process
from tqdm.notebook import tqdm

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('ph_pipeline')
log.info('Imports OK')


15:47:13  INFO      Imports OK


## Cell 3 — Load reference data

Loads `dim_location` (42 000+ PSGC barangay rows) and `ph_address_alias` (499 alias pairs), then builds fast in-memory lookup lists for fuzzy matching.

In [71]:
def _read_csv_with_fallback(path: str) -> pd.DataFrame:
    strict_encodings = ['utf-8', 'utf-8-sig']
    for enc in strict_encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue

    # Keep mostly-UTF8 files readable even if they contain a few broken bytes.
    try:
        log.warning(f'Loaded {path} using utf-8 with replacement for invalid bytes')
        return pd.read_csv(path, encoding='utf-8', encoding_errors='replace')
    except Exception:
        pass

    for enc in ['cp1252', 'latin1']:
        try:
            log.warning(f'Loaded {path} using fallback encoding: {enc}')
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue

    raise ValueError(f'Unable to decode CSV file: {path}')


def load_reference(dim_path: str, alias_path: str):
    log.info('Loading dim_location ...')
    dim = _read_csv_with_fallback(dim_path)
    str_cols = ['region_name', 'province_name', 'city_name', 'barangay_name']
    for c in str_cols:
        dim[c] = dim[c].fillna('').astype(str).str.upper().str.strip()

    log.info('Loading alias table ...')
    alias = _read_csv_with_fallback(alias_path)
    alias['pattern'] = alias['pattern'].fillna('').astype(str).str.upper().str.strip()
    alias['replacement'] = alias['replacement'].fillna('').astype(str).str.upper().str.strip()

    cities = sorted(x for x in dim['city_name'].unique().tolist() if x)
    provinces = sorted(x for x in dim['province_name'].unique().tolist() if x)
    regions = sorted(x for x in dim['region_name'].unique().tolist() if x)
    barangays = sorted(x for x in dim['barangay_name'].unique().tolist() if x)

    log.info(
        f'dim_location: {len(dim):,} rows | '
        f'{len(cities)} cities | {len(provinces)} provinces | '
        f'{len(regions)} regions | {len(barangays):,} barangays'
    )
    return dim, alias, cities, provinces, regions, barangays


dim, alias_df, cities, provinces, regions, barangays = load_reference(
    DIM_LOCATION, ALIAS_FILE
)

print('\n-- Sample dim_location --')
display(dim.head(3))
print('\n-- Sample aliases --')
display(alias_df.head(10))


15:47:13  INFO      Loading dim_location ...
15:47:13  WARNING   Loaded ..\..\data\mapping\dim_location_20260316_v2.csv using utf-8 with replacement for invalid bytes
15:47:13  INFO      Loading alias table ...
15:47:13  WARNING   Loaded ..\..\data\utils\ph_address_alias_extended_v4.csv using utf-8 with replacement for invalid bytes
15:47:13  INFO      dim_location: 42,011 rows | 1426 cities | 83 provinces | 18 regions | 25,843 barangays



-- Sample dim_location --


,psgc_10_digit,region_name,province_name,city_name,barangay_name,region_code,province_code,city_code,barangay_code
0,1400101001,CORDILLERA ADMINISTRATIVE REGION (CAR),ABRA,BANGUED,AGTANGAO,14,1,1,1
1,1400101002,CORDILLERA ADMINISTRATIVE REGION (CAR),ABRA,BANGUED,ANGAD,14,1,1,2
2,1400101003,CORDILLERA ADMINISTRATIVE REGION (CAR),ABRA,BANGUED,BAÑACAO,14,1,1,3



-- Sample aliases --


,pattern,replacement
0,BRGY,BARANGAY
1,BRGY.,BARANGAY
2,BRG,BARANGAY
3,BRG.,BARANGAY
4,BARRIO,BARANGAY
5,BARRIO.,BARANGAY
6,POB,POBLACION
7,POB.,POBLACION
8,POBL,POBLACION
9,POBL.,POBLACION


## Cell 4 — Stage 1: Alias normalization

Builds a **single-pass compiled regex** from all 499 alias pairs (sorted longest-first so multi-word patterns like `CITY OF MARIKINA` fire before the single-token `MARIKINA`).

A post-pass deduplication step catches chained-alias artifacts like `CITY CITY` or `BARANGAY BARANGAY` that arise when both `CITY OF X → X CITY` and `X → X CITY` fire on the same string.

In [72]:
def build_alias_regex(alias_df: pd.DataFrame):
    pairs = [
        (p.strip(), r.strip())
        for p, r in zip(alias_df['pattern'], alias_df['replacement'])
        if isinstance(p, str) and p.strip()
    ]
    pairs.sort(key=lambda x: -len(x[0]))   # longest pattern first
    replace_map = {p: r for p, r in pairs}
    pattern  = '|'.join(re.escape(p) for p, _ in pairs)
    compiled = re.compile(r'\b(' + pattern + r')\b')
    return compiled, replace_map


def normalize_alias(text: str, compiled_re, replace_map: dict) -> str:
    text = str(text).upper().strip()
    # Normalize punctuation first so aliases like 'PQUE.' can match robustly.
    text = re.sub(r'[\.,;:/\\\-\(\)\[\]\{\}]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = compiled_re.sub(lambda m: replace_map.get(m.group(0), m.group(0)), text)
    # Remove duplicate consecutive words (e.g. 'CITY CITY', 'BARANGAY BARANGAY')
    text = re.sub(
        r'\b(CITY|BARANGAY|STREET|AVENUE|BOULEVARD|VILLAGE|PROVINCE)\s+\1\b',
        r'\1', text, flags=re.IGNORECASE
    )
    return re.sub(r'\s+', ' ', text).strip()


compiled_re, replace_map = build_alias_regex(alias_df)
log.info(f'Alias regex built from {len(alias_df)} patterns')

# -- Quick test on sample addresses --------------------------------------------
test_cases = [
    'sapang palay sjdm',
    '112 Ballecer st Zone South Signal Village Taguig',
    '55 Dawn St South Kim View Park SSS Village Concepcionh 2 MArikina City',
    'loilo City',
    'carmencdoc Carmen CAGAYAN DE ORO CITY (Capital) MISAMIS ORIENTAL',
    'LIONS PARK RES SUNVALLEY PQUE.',
]
print(f'{"ORIGINAL":<55}  {"NORMALIZED"}')
print('-' * 110)
for t in test_cases:
    print(f'{t:<55}  {normalize_alias(t, compiled_re, replace_map)}')


15:47:13  INFO      Alias regex built from 512 patterns


ORIGINAL                                                 NORMALIZED
--------------------------------------------------------------------------------------------------------------
sapang palay sjdm                                        SAPANG PALAY SAN JOSE DEL MONTE CITY
112 Ballecer st Zone South Signal Village Taguig         112 BALLECER STREET ZONE SOUTH SIGNAL VILLAGE TAGUIG CITY
55 Dawn St South Kim View Park SSS Village Concepcionh 2 MArikina City  55 DAWN STREET SOUTH KIM VIEW PARK SSS VILLAGE CONCEPCIONH 2 MARIKINA CITY
loilo City                                               LOILO CITY
carmencdoc Carmen CAGAYAN DE ORO CITY (Capital) MISAMIS ORIENTAL  CARMENCDOC CARMEN CAGAYAN DE ORO CITY CAPITAL MISAMIS ORIENTAL
LIONS PARK RES SUNVALLEY PQUE.                           LIONS PARK RES SUNVALLEY PARA�AQUE CITY


## Cell 5 — Stage 2: Fuzzy matching helpers

Two functions used throughout the pipeline:
- **`best_match`** — matches a single query string against a list (full-string scorer)
- **`token_scan`** — splits the address into tokens and tries each token independently; returns the highest-scoring hit across all tokens

Both use `rapidfuzz.fuzz.WRatio` which handles partial overlaps well (e.g. `QUEZON CITY` inside a longer address string).

In [73]:
def best_match(
    query: str,
    choices: list,
    scorer=fuzz.WRatio,
    score_cutoff: int = 0,
) -> tuple[Optional[str], int]:
    """Return (best_match_string, score) or (None, 0) if below cutoff."""
    if not query:
        return None, 0
    result = process.extractOne(query, choices, scorer=scorer,
                                score_cutoff=score_cutoff)
    return (result[0], int(result[1])) if result else (None, 0)


def token_scan(
    tokens: list,
    choices: list,
    score_cutoff: int = 0,
) -> tuple[Optional[str], int]:
    """Try each token against choices; return best (match, score)."""
    best_s, best_m = 0, None
    for tok in tokens:
        if len(tok) < 3:
            continue
        m, s = best_match(tok, choices, score_cutoff=score_cutoff)
        if s > best_s:
            best_s, best_m = s, m
    return best_m, best_s


# ── Quick smoke test ──────────────────────────────────────────────────────────
tests = [
    ('QUEZON CITY', cities),
    ('ILOCOS SUR', provinces),
    ('TAGUIG', cities),
    ('SOUTH SIGNAL', cities),   # should score low / return nothing meaningful
]
print(f'{"Query":<25}  {"Best match":<35}  Score')
print('─' * 75)
for q, lst in tests:
    m, s = best_match(q, lst, score_cutoff=0)
    print(f'{q:<25}  {str(m):<35}  {s}')


Query                      Best match                           Score
───────────────────────────────────────────────────────────────────────────
QUEZON CITY                QUEZON CITY                          100
ILOCOS SUR                 ILOCOS SUR                           100
TAGUIG                     CITY OF TAGUIG                       90
SOUTH SIGNAL               SIGAY                                72


## Cell 6 — Stage 2 + 3: Core address matching & confidence gate

### Strict city-required rule
An address must contain explicit city evidence after normalization (`ph_address_alias` + raw text).
If no city evidence is present, the row is forced invalid and goes to invalid output.

### Area-based city exceptions (explicitly allowed)
Some place cues are treated as city evidence even without the word `CITY`:
- **Quezon City**: `AURORA BLVD`, `CUBAO`, `NEW MANILA`, `KAMUNING`, `TIMOG`, `DILIMAN`, `KATIPUNAN`, `COMMONWEALTH`
- **Taguig**: `BGC`, `BONIFACIO GLOBAL CITY`, `FORT BONIFACIO`, `MCKINLEY HILL`, `MCKINLEY WEST`
- **Manila districts**: district terms (e.g., `TONDO`, `BINONDO`, `ERMITA`) are resolved to matching `city_name` values from `dim_location` (no forced `CITY OF MANILA`)

### Confidence levels
| Level | Condition | Action |
|---|---|---|
| `high` | city score ≥ 88 | Accept immediately — no API |
| `medium` | city score 65–87 | Send to Nominatim for verification |
| `low` | missing/weak city evidence | Mark invalid (or API if enabled) |
| `none` | nothing matched | Mark invalid |

In [74]:
def match_address(
    address_norm: str,
    dim: pd.DataFrame,
    cities: list, provinces: list, regions: list, barangays: list,
    city_high: int = CITY_SCORE_HIGH,
    city_low: int  = CITY_SCORE_LOW,
    prov_high: int = PROV_SCORE_HIGH,
    prov_low: int  = PROV_SCORE_LOW,
) -> dict:
    result = dict(
        city_name=None, city_code=None, city_score=0,
        province_name=None, province_code=None, province_score=0,
        region_name=None, region_code=None,
        barangay_name=None, barangay_code=None,
        psgc_10_digit=None,
        confidence='none', needs_api=False,
        city_required_pass=False, city_signal=None, invalid_reason=None,
    )

    tokens = [t for t in re.split(r'[,\s]+', address_norm) if len(t) >= 3]

    def _ascii_fold(text: str) -> str:
        # Fold accents so PARAÑAQUE and PARANAQUE match consistently.
        import unicodedata
        txt = unicodedata.normalize('NFKD', str(text))
        return ''.join(ch for ch in txt if not unicodedata.combining(ch))

    def _canon(text: str) -> str:
        text = _ascii_fold(str(text).upper())
        text = re.sub(r'[^A-Z0-9]+', ' ', text)
        return re.sub(r'\s+', ' ', text).strip()

    def _build_city_variant_map(city_list: list) -> dict[str, str]:
        variant_map = {}
        for city_name in city_list:
            city_raw = str(city_name).strip()
            if not city_raw:
                continue

            base = _canon(city_raw)
            variants = {base}

            if base.startswith('CITY OF '):
                core = base[len('CITY OF '):].strip()
                if core:
                    variants.add(core)
                    variants.add(f'{core} CITY')
            if base.startswith('MUNICIPALITY OF '):
                core = base[len('MUNICIPALITY OF '):].strip()
                if core:
                    variants.add(core)
            if base.endswith(' CITY'):
                core = base[:-len(' CITY')].strip()
                if core:
                    variants.add(core)
                    variants.add(f'CITY OF {core}')

            for v in variants:
                if v and v not in variant_map:
                    variant_map[v] = city_name
        return variant_map

    MANILA_DISTRICT_CORES = [
        'TONDO', 'BINONDO', 'ERMITA', 'INTRAMUROS', 'MALATE', 'PACO',
        'PANDACAN', 'PORT AREA', 'QUIAPO', 'SAMPALOC', 'SAN NICOLAS',
        'SANTA ANA', 'SANTA CRUZ',
    ]

    def _build_manila_district_map(city_list: list) -> dict[str, list[str]]:
        district_map = {core: [] for core in MANILA_DISTRICT_CORES}
        for city_name in city_list:
            city_can = _canon(city_name)
            for core in MANILA_DISTRICT_CORES:
                core_can = _canon(core)
                if f' {core_can} ' in f' {city_can} ':
                    district_map[core].append(city_name)

        # Keep only district keys that really exist in dim_location cities.
        return {
            k: sorted(set(v))
            for k, v in district_map.items()
            if v
        }

    if not hasattr(match_address, '_cache'):
        city_variant_map = _build_city_variant_map(cities)
        sorted_city_variants = sorted(city_variant_map.keys(), key=len, reverse=True)
        barangay_canon = {_canon(b) for b in barangays if str(b).strip()}
        manila_district_map = _build_manila_district_map(cities)
        match_address._cache = {
            'city_variant_map': city_variant_map,
            'sorted_city_variants': sorted_city_variants,
            'barangay_canon': barangay_canon,
            'manila_district_map': manila_district_map,
        }

    city_variant_map = match_address._cache['city_variant_map']
    sorted_city_variants = match_address._cache['sorted_city_variants']
    barangay_canon = match_address._cache['barangay_canon']
    manila_district_map = match_address._cache['manila_district_map']
    manila_city_values = {v for vals in manila_district_map.values() for v in vals}

    def _find_explicit_city(text_canon: str) -> Optional[str]:
        padded = f' {text_canon} '
        for variant in sorted_city_variants:
            if f' {variant} ' in padded:
                return city_variant_map[variant]
        return None

    city_area_exception_terms = {
        'QUEZON CITY': [
            'AURORA BLVD', 'AURORA BOULEVARD', 'CUBAO', 'NEW MANILA',
            'KAMUNING', 'TIMOG', 'DILIMAN', 'KATIPUNAN', 'COMMONWEALTH',
            'TOMAS MORATO',
        ],
        'CITY OF TAGUIG': [
            'BGC', 'BONIFACIO GLOBAL CITY', 'FORT BONIFACIO',
            'MCKINLEY HILL', 'MCKINLEY WEST', 'UPTOWN BGC',
        ],
    }

    lookup_text = _canon(address_norm)
    explicit_city = _find_explicit_city(lookup_text)

    exception_city = None
    for exc_city, terms in city_area_exception_terms.items():
        if any(_canon(t) in lookup_text for t in terms):
            exception_city = exc_city
            break

    padded_lookup = f' {lookup_text} '
    manila_hits = []
    for core, city_candidates in manila_district_map.items():
        core_can = _canon(core)
        if f' {core_can} ' in padded_lookup:
            if len(city_candidates) == 1:
                manila_hits.append(city_candidates[0])
            else:
                # If multiple dim_location city names share a core (e.g. I/II),
                # first try explicit candidate mention; then fallback to fuzzy city pick.
                explicit_city_hits = []
                for c in city_candidates:
                    if f' {_canon(c)} ' in padded_lookup:
                        explicit_city_hits.append(c)
                if explicit_city_hits:
                    manila_hits.extend(explicit_city_hits)
                else:
                    fuzzy_city, fuzzy_score = best_match(lookup_text, city_candidates, score_cutoff=70)
                    if fuzzy_city:
                        manila_hits.append(fuzzy_city)

    manila_hits = sorted(set(manila_hits))
    district_city = manila_hits[0] if len(manila_hits) == 1 else None

    if district_city and (
        not explicit_city or _canon(explicit_city) in {'MANILA', 'CITY OF MANILA'}
    ):
        city_signal = district_city
        city_signal_source = 'dim_mnl_district'
    elif exception_city and explicit_city not in {exception_city}:
        city_signal = exception_city
        city_signal_source = 'area_exception'
    elif explicit_city:
        city_signal = explicit_city
        city_signal_source = 'explicit_city'
    else:
        city_signal = None
        city_signal_source = None

    has_city_keyword = bool(re.search(r'\b(CITY|MUNICIPALITY|MUN)\b', lookup_text))
    if city_signal_source == 'explicit_city' and not has_city_keyword:
        sig_canon = _canon(city_signal) if city_signal else ''
        is_manila_district_city = city_signal in manila_city_values
        if sig_canon in barangay_canon and not exception_city and not is_manila_district_city:
            city_signal = None
            city_signal_source = None

    prov_hint, prov_hint_score = token_scan(tokens, provinces, score_cutoff=prov_low)
    city_full_hint, city_full_score = best_match(address_norm, cities, score_cutoff=city_low)
    city_tok_hint, city_tok_score = token_scan(tokens, cities, score_cutoff=city_low)
    city_hint, city_hint_score = (
        (city_full_hint, city_full_score)
        if city_full_score >= city_tok_score
        else (city_tok_hint, city_tok_score)
    )

    def _name_overlaps_address(name: str, text: str) -> bool:
        skip = {'CITY', 'MUNICIPALITY', 'OF', 'PROVINCE'}
        words = [w for w in str(name).split() if w not in skip and len(w) >= 4]
        return any(_canon(w) in _canon(text) for w in words)

    generic_brgy_terms = {
        'PUROK', 'POBLACION', 'ZONE', 'CENTRO', 'PROPER',
        'BARANGAY', 'SITIO', 'VILLA', 'DISTRICT', 'AREA'
    }

    def _city_signal_matches(candidate_city: str) -> bool:
        if city_signal_source in {'explicit_city', 'area_exception', 'dim_mnl_district'}:
            return candidate_city == city_signal
        return False

    def _best_barangay_row(brgy_name: Optional[str], brgy_raw_score: int, source: str):
        if not brgy_name:
            return None, -1, -1
        subset = dim[dim['barangay_name'] == brgy_name]
        if subset.empty:
            return None, -1, -1

        best_row = None
        best_adjusted = -1
        best_bonus = -1
        for _, row in subset.iterrows():
            bonus = 0
            if city_hint and row['city_name'] == city_hint:
                bonus += 25
            if prov_hint and row['province_name'] == prov_hint:
                bonus += 20
            if _name_overlaps_address(row['city_name'], address_norm):
                bonus += 10
            if _name_overlaps_address(row['province_name'], address_norm):
                bonus += 8

            adjusted = brgy_raw_score + bonus
            if adjusted > best_adjusted:
                best_adjusted = adjusted
                best_bonus = bonus
                best_row = row

        brgy_upper = str(brgy_name).upper().strip()
        brgy_words = [w for w in brgy_upper.split() if w]
        is_generic = brgy_upper in generic_brgy_terms
        is_short_single = len(brgy_words) == 1 and len(brgy_upper) < 7
        looks_placeholder = brgy_upper.startswith('BARANG')

        if source == 'token' and best_bonus < 10:
            return None, -1, -1
        if is_generic and best_bonus < 15:
            return None, -1, -1
        if (is_short_single or looks_placeholder) and best_bonus < 20:
            return None, -1, -1
        if len(brgy_words) == 1 and source == 'full' and best_bonus < 10:
            return None, -1, -1

        return best_row, best_adjusted, best_bonus

    brgy_full, brgy_score_full = best_match(address_norm, barangays, score_cutoff=BRGY_SCORE_MIN)
    brgy_tok,  brgy_score_tok  = token_scan(tokens, barangays, score_cutoff=BRGY_SCORE_MIN)

    row_full, adj_full, bonus_full = _best_barangay_row(brgy_full, brgy_score_full, source='full')
    row_tok,  adj_tok,  bonus_tok  = _best_barangay_row(brgy_tok, brgy_score_tok, source='token')

    if max(adj_full, adj_tok) >= BRGY_SCORE_MIN:
        brgy_row = row_full if adj_full >= adj_tok else row_tok
        if brgy_row is not None and _city_signal_matches(brgy_row['city_name']):
            result.update(
                city_name=brgy_row['city_name'],
                city_code=int(brgy_row['city_code']),
                city_score=95,
                province_name=brgy_row['province_name'],
                province_code=int(brgy_row['province_code']),
                province_score=95,
                region_name=brgy_row['region_name'],
                region_code=int(brgy_row['region_code']),
                barangay_name=brgy_row['barangay_name'],
                barangay_code=int(brgy_row['barangay_code']),
                psgc_10_digit=int(brgy_row['psgc_10_digit']),
                confidence='high', needs_api=False,
                city_required_pass=True, city_signal=city_signal, invalid_reason=None,
            )
            return result

    prov_match, prov_score = prov_hint, prov_hint_score

    if city_signal_source == 'explicit_city':
        city_match, city_score = city_signal, 100
    elif city_signal_source in {'area_exception', 'dim_mnl_district'}:
        city_match, city_score = city_signal, 95
    else:
        city_match, city_score = None, 0

    if city_match:
        candidate_provs = dim[dim['city_name'] == city_match]['province_name'].unique()
        if len(candidate_provs) > 1:
            if not (prov_match and prov_match in candidate_provs
                    and prov_score >= prov_low):
                city_score = 0
                city_match = None

    if city_match:
        subset = dim[dim['city_name'] == city_match]
        if prov_match and prov_score >= prov_low:
            sub2 = subset[subset['province_name'] == prov_match]
            if not sub2.empty:
                subset = sub2
        if not subset.empty:
            first = subset.iloc[0]
            result.update(
                city_name=city_match, city_code=int(first['city_code']),
                city_score=city_score,
                province_name=first['province_name'], province_code=int(first['province_code']),
                province_score=max(prov_score, 90 if city_signal_source in {'area_exception', 'dim_mnl_district'} else prov_score),
                region_name=first['region_name'], region_code=int(first['region_code']),
            )
        else:
            # Keep city text when evidence exists but dim_location cannot resolve a canonical row.
            result.update(city_name=city_match, city_score=max(city_score, city_low))
    elif prov_match and prov_score >= prov_high:
        subset = dim[dim['province_name'] == prov_match]
        first = subset.iloc[0]
        result.update(
            province_name=prov_match, province_code=int(first['province_code']),
            province_score=prov_score,
            region_name=first['region_name'], region_code=int(first['region_code']),
        )

    result['city_signal'] = city_signal
    # Core rule: a row is city-valid when city evidence exists in the address.
    result['city_required_pass'] = bool(city_signal_source)

    if not city_signal_source:
        result['invalid_reason'] = 'missing_explicit_city'
        result['confidence'], result['needs_api'] = 'low', False
        return result
    if not result['city_name']:
        result['invalid_reason'] = 'city_not_resolved'
        result['confidence'], result['needs_api'] = 'low', True
        return result

    cs = result['city_score']
    ps = result['province_score']
    if cs >= city_high:
        result['confidence'], result['needs_api'] = 'high', False
    elif cs >= city_low:
        result['confidence'], result['needs_api'] = 'medium', True
    elif result['province_name'] and ps >= prov_high:
        result['confidence'], result['needs_api'] = 'medium', True
    else:
        result['confidence'], result['needs_api'] = 'low', True

    result['invalid_reason'] = None
    return result


# -- Generalized regression tests (not single-case tuning) --------------------
test_addresses = [
    '19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY',
    '112 Ballecer st Zone South Signal Village Taguig',
    'LIONS PARK RES SUNVALLEY PQUE.',
    'House 4 Adolfo Compound,Highway 11 St Brgy Talamban, City of Cebu',
    'BGC 30TH STREET',
    'TONDO MANILA',
    'BINONDO MANILA',
    '2248 Angel Linao St',
    '1206 Don Quiote St',
    '1067 UNIT 7 ARLEGUI ST',
    '122 baltazar st 10th ave',
    'AURORA BLVD COR 20TH AVE',
    'NEW MANILA near E. Rodriguez',
]

for idx, raw in enumerate(test_addresses, 1):
    addr = normalize_alias(raw, compiled_re, replace_map)
    ans = match_address(addr, dim, cities, provinces, regions, barangays)
    print(f'Test {idx}')
    print(f'  Input      : {raw}')
    print(f'  Normalized : {addr}')
    print(f'  City signal: {ans["city_signal"]}')
    print(f'  City       : {ans["city_name"]}')
    print(f'  Province   : {ans["province_name"]}')
    print(f'  Barangay   : {ans["barangay_name"]}')
    print(f'  Confidence : {ans["confidence"]}')
    print(f'  Invalid rsn: {ans["invalid_reason"]}')
    print('-' * 80)

Test 1
  Input      : 19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY
  Normalized : 19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY
  City signal: QUEZON CITY
  City       : QUEZON CITY
  Province   : METRO MANILA
  Barangay   : BATASAN HILLS
  Confidence : high
  Invalid rsn: None
--------------------------------------------------------------------------------
Test 2
  Input      : 112 Ballecer st Zone South Signal Village Taguig
  Normalized : 112 BALLECER STREET ZONE SOUTH SIGNAL VILLAGE TAGUIG CITY
  City signal: CITY OF TAGUIG
  City       : CITY OF TAGUIG
  Province   : METRO MANILA
  Barangay   : SOUTH SIGNAL VILLAGE
  Confidence : high
  Invalid rsn: None
--------------------------------------------------------------------------------
Test 3
  Input      : LIONS PARK RES SUNVALLEY PQUE.
  Normalized : LIONS PARK RES SUNVALLEY PARA�AQUE CITY
  City signal: CITY OF PARA�AQUE
  City       : CITY OF PARA�AQUE
  Province   : METRO MANILA
  Barangay   : SUN V

## Cell 7 — Run Stage 1–3 on all input rows

Loads the input file, applies alias normalization, then runs the fuzzy matcher on every row. Results are split into `high_conf` (no API needed) and `low_conf` (needs Nominatim verification).

In [75]:
import time

t_start = time.time()

if input_paths:
    existing_paths = [p for p in input_paths if p.exists()]
    missing_paths = [p for p in input_paths if not p.exists()]

    for mp in missing_paths:
        log.warning(f'Missing batch file: {mp}')

    if not existing_paths:
        raise FileNotFoundError('No batch files found in input_paths.')

    log.info(f'Loading batch files: {len(existing_paths)} found')
    frames = []
    for p in existing_paths:
        tmp = pd.read_excel(p)
        tmp['_batch_file'] = p.name
        frames.append(tmp)
    df_input = pd.concat(frames, ignore_index=True)
else:
    log.info(f'Loading {INPUT_FILE} ...')
    df_input = pd.read_excel(INPUT_FILE)

if MAX_ROWS:
    df_input = df_input.iloc[:MAX_ROWS]
    log.info(f'Limiting to {MAX_ROWS} rows (MAX_ROWS is set)')

addr_col = df_input.columns[0]
log.info(f'Address column: "{addr_col}"  |  Total rows: {len(df_input):,}')

# -- Stage 1: alias normalization ---------------------------------------------
log.info('Stage 1: alias normalization ...')
addresses = df_input[addr_col].fillna('').astype(str).tolist()
normalized = [normalize_alias(x, compiled_re, replace_map) for x in addresses]
df_input['address_normalized'] = normalized

# -- Stage 2-3: fuzzy match + confidence gate ---------------------------------
log.info('Stage 2-3: fuzzy matching + confidence gate ...')
records = []
batch_files = df_input['_batch_file'].tolist() if '_batch_file' in df_input.columns else None

for i, addr_norm in enumerate(tqdm(normalized, total=len(normalized), desc='Fuzzy match', unit='row')):
    rec = {
        'original_address': addresses[i],
        'address_normalized': addr_norm,
        'api_status': 'skipped',
        'osm_city': None, 'osm_province': None, 'osm_region': None,
        'osm_display': None, 'osm_lat': None, 'osm_lon': None,
    }
    if batch_files is not None:
        rec['batch_file'] = batch_files[i]
    rec.update(match_address(
        addr_norm,
        dim, cities, provinces, regions, barangays,
    ))
    rec['needs_api'] = False
    records.append(rec)

high_conf = [r for r in records if r.get('confidence') == 'high']
low_conf = [r for r in records if r.get('confidence') != 'high']

t_fuzzy = time.time() - t_start
log.info(f'Fuzzy pass done in {t_fuzzy:.1f}s')
log.info(f'  High confidence : {len(high_conf):,}')
log.info(f'  Lower confidence: {len(low_conf):,}')
log.info('  API stage is disabled; fuzzy-only output will be used')

# -- Preview ------------------------------------------------------------------
preview_cols = [
    'original_address', 'address_normalized',
    'city_name', 'province_name', 'confidence', 'city_score'
]
if records and 'batch_file' in records[0]:
    preview_cols = ['batch_file'] + preview_cols

df_preview = pd.DataFrame(records)[preview_cols]
display(df_preview)


15:47:16  INFO      Loading batch files: 50 found
15:47:21  INFO      Address column: "Original_Address"  |  Total rows: 50,000
15:47:21  INFO      Stage 1: alias normalization ...
15:47:22  INFO      Stage 2-3: fuzzy matching + confidence gate ...


Fuzzy match:   0%|          | 0/50000 [00:00<?, ?row/s]

KeyboardInterrupt: 

In [ ]:
# Snapshot fuzzy outputs before API mutates records
pre_api_df = pd.DataFrame([dict(r) for r in records])
print(f'Captured pre_api_df rows: {len(pre_api_df):,}')

Captured pre_api_df rows: 10,000


## Cell 8 -- API stage disabled

API verification is intentionally disabled in this fast run profile.

This notebook now runs in fuzzy-only mode using:
- `dim_location` for PSGC hierarchy mapping
- `ph_address_alias_extended_v4.csv` for normalization

If API is needed later, re-enable it in configuration and restore the API stage logic.


In [ ]:
# API stage intentionally disabled for maximum throughput in fuzzy-only mode.
# Keep this cell as a no-op so downstream cells continue to run without changes.

USE_API = False
log.info('API stage skipped (fuzzy-only mode).')

for row in low_conf:
    row['api_status'] = 'skipped'
    row['needs_api'] = False

log.info('Proceeding directly to merge/output stages.')


13:12:01  INFO      API stage skipped (fuzzy-only mode).
13:12:01  INFO      Proceeding directly to merge/output stages.


## Cell 9 — Stage 5a: Merge results & determine validity

Combines all processed rows and applies the final validity rule.

Final rule in this notebook profile:
- Address is **valid** when city evidence is present in the address (`city_required_pass=True`).
- Rows without city evidence are invalid (`missing_explicit_city`).
- If city evidence exists but dim mapping is incomplete, row remains valid and `invalid_reason` can still show `city_not_resolved` for diagnostics.

This enforces the no-assumption rule while keeping city-present rows out of pure-invalid false negatives.

In [ ]:
all_records = low_conf if USE_API else (high_conf + low_conf)
out_df = pd.DataFrame(all_records)

if 'city_required_pass' not in out_df.columns:
    out_df['city_required_pass'] = False
out_df['city_required_pass'] = out_df['city_required_pass'].fillna(False).astype(bool)

# Final validity flag with strict city-evidence requirement
if USE_API:
    out_df['is_valid'] = out_df['city_required_pass']
else:
    out_df['is_valid'] = out_df['city_required_pass']

# Confidence value counts
print('Confidence distribution:')
print(out_df['confidence'].value_counts().to_string())
if 'api_status' in out_df.columns:
    print('\nAPI status distribution:')
    print(out_df['api_status'].value_counts().to_string())
if 'invalid_reason' in out_df.columns:
    print('\nInvalid reason distribution:')
    print(out_df['invalid_reason'].fillna('resolved').value_counts().to_string())
print(f'\nValid   : {out_df["is_valid"].sum():,}')
print(f'Invalid : {(~out_df["is_valid"]).sum():,}')
print(f'Total   : {len(out_df):,}')


Confidence distribution:
confidence
low     9380
high     620

API status distribution:
api_status
skipped    10000

Invalid reason distribution:
invalid_reason
missing_explicit_city    9342
resolved                  620
city_not_resolved          38

Valid   : 658
Invalid : 9,342
Total   : 10,000


In [ ]:
# City-only sanity check: barangay should remain blank when not explicit
city_only_mask = out_df['original_address'].astype(str).str.lower().isin(['vigan ilocos sur', 'loilo city'])
display(out_df.loc[city_only_mask, [
    'original_address', 'city_name', 'province_name', 'barangay_name', 'barangay_code'
]])

,original_address,city_name,province_name,barangay_name,barangay_code


In [ ]:
# Detailed-address check: barangay should still be populated when appropriate
detail_mask = out_df['original_address'].astype(str).str.contains(
    'Batasan|South Signal|Kim View|sapang palay', case=False, na=False
)
display(out_df.loc[detail_mask, [
    'original_address', 'city_name', 'province_name', 'barangay_name', 'barangay_code'
]])

,original_address,city_name,province_name,barangay_name,barangay_code
911,"13 Chase St Filinvest 2, Batasan Hills",NaN,BATAAN,NaN,NaN
1292,197 Lot B Sapang palay proper,NaN,NaN,NaN,NaN
1330,Rosas St. Batasan Hills,NaN,BATAAN,NaN,NaN
1628,128 Sapang palay Proper CSJDM,NaN,NaN,NaN,NaN
2746,112 SINAGTALA ST BATASAN HILLS,NaN,BATAAN,NaN,NaN
3987,FRIENDSHIP ST ZONE 6 SOUTH SIGNAL,NaN,SOUTH COTABATO,NaN,NaN
4114,Sapang palay proper,NaN,NaN,NaN,NaN
5275,BLK 16 LOT 23 HACIENDA ST SUGARTOWN HOMES BATA...,NaN,BATAAN,NaN,NaN
5559,142 Srgt. Ignacio St (Sapang Palay),NaN,NaN,NaN,NaN
5623,BATASANG PAMBANSA COMPLEX,NaN,NaN,NaN,NaN


In [ ]:
# Before-vs-after change report (fuzzy baseline vs API-final)
compare_cols = ['city_name', 'province_name', 'confidence', 'api_status']
base = pre_api_df[['original_address'] + compare_cols].copy()
base = base.rename(columns={
    'city_name': 'city_before',
    'province_name': 'province_before',
    'confidence': 'confidence_before',
    'api_status': 'api_status_before',
})

final = out_df[['original_address'] + compare_cols].copy()
final = final.rename(columns={
    'city_name': 'city_after',
    'province_name': 'province_after',
    'confidence': 'confidence_after',
    'api_status': 'api_status_after',
})

change_df = base.merge(final, on='original_address', how='inner')
changed_mask = (
    change_df['city_before'].fillna('') != change_df['city_after'].fillna('')
) | (
    change_df['province_before'].fillna('') != change_df['province_after'].fillna('')
) | (
    change_df['confidence_before'].fillna('') != change_df['confidence_after'].fillna('')
) | (
    change_df['api_status_before'].fillna('') != change_df['api_status_after'].fillna('')
)

changed_rows = change_df[changed_mask].reset_index(drop=True)
print(f'Rows changed after API verification: {len(changed_rows):,} / {len(change_df):,}')
display(changed_rows)

Rows changed after API verification: 0 / 10,000


,original_address,city_before,province_before,confidence_before,api_status_before,city_after,province_after,confidence_after,api_status_after


In [ ]:
# Compact delta view
compact = changed_rows[[
    'original_address',
    'city_before', 'city_after',
    'province_before', 'province_after',
    'api_status_before', 'api_status_after'
]].copy()
compact['original_address'] = compact['original_address'].astype(str).str.slice(0, 70)
display(compact)

# Highlight the user's example row
mask = changed_rows['original_address'].astype(str).str.contains('General Espino South Signal', case=False, na=False)
if mask.any():
    print('General Espino South Signal change:')
    display(changed_rows.loc[mask, [
        'original_address',
        'city_before', 'city_after',
        'province_before', 'province_after',
        'confidence_before', 'confidence_after',
        'api_status_before', 'api_status_after'
    ]])

,original_address,city_before,city_after,province_before,province_after,api_status_before,api_status_after


## Cell 10 — Stage 5b: Export to Excel

Creates three output files per batch using only the batch suffix:
- `output/verified/verified_addresses_parts_011_020.xlsx`
- `output/invalid/invalid_addresses_parts_011_020.xlsx`
- `output/combined_addresses_parts_011_020.xlsx`

Rules:
- Verified/Invalid files use only the base address hierarchy columns in the required order.
- Combined file includes `Normalized_Address` plus base columns, PSGC/confidence/scoring/OSM diagnostics, and a `status` column.
- Different batch ranges produce different filenames, so previous batch outputs are preserved.

In [ ]:
from pathlib import Path

# -- Output targets ------------------------------------------------------------
out_root = Path(BASE_DIR) / 'output'
out_verified_dir = out_root / 'verified'
out_invalid_dir = out_root / 'invalid'
out_verified_dir.mkdir(parents=True, exist_ok=True)
out_invalid_dir.mkdir(parents=True, exist_ok=True)

# Build suffix from detected input parts (e.g. parts_001_010)
batch_nums = []
for p in input_paths:
    m = re.search(r'part_(\d+)', p.name.lower())
    if m:
        batch_nums.append(int(m.group(1)))

if batch_nums:
    batch_suffix = f'parts_{min(batch_nums):03d}_{max(batch_nums):03d}'
elif input_paths:
    batch_suffix = f'batch_{len(input_paths):03d}'
else:
    batch_suffix = 'single'

VERIFIED_FILE = str(out_verified_dir / f'verified_addresses_{batch_suffix}.xlsx')
INVALID_FILE = str(out_invalid_dir / f'invalid_addresses_{batch_suffix}.xlsx')
COMBINED_FILE = str(out_root / f'combined_addresses_{batch_suffix}.xlsx')

# -- Column order required by business template (image) -----------------------
BASE_EXPORT_COLS = [
    'Original_Address',
    'barangay_code', 'barangay',
    'city_code', 'city',
    'province_code', 'province',
    'region_code', 'region_name',
]

# Build standardized base table from pipeline output
base_df = out_df.copy()
base_df = base_df.rename(columns={
    'original_address': 'Original_Address',
    'barangay_name': 'barangay',
    'city_name': 'city',
    'province_name': 'province',
})

# Ensure expected columns exist even when null in invalid rows
for c in BASE_EXPORT_COLS:
    if c not in base_df.columns:
        base_df[c] = None

verified_df = base_df[base_df['is_valid']][BASE_EXPORT_COLS].reset_index(drop=True)
invalid_df = base_df[~base_df['is_valid']][BASE_EXPORT_COLS].reset_index(drop=True)

# Combined file keeps additional diagnostics + status
combined_cols = [
    'Original_Address',
    'Normalized_Address',
    'barangay_code', 'barangay',
    'city_code', 'city',
    'province_code', 'province',
    'region_code', 'region_name',
    'psgc_10',
    'confidence',
    'city_score',
    'province_score',
    'osm_display',
    'osm_lat',
    'osm_lon',
    'status',
]

combined_df = base_df.copy()
combined_df['Normalized_Address'] = combined_df.get('address_normalized')
combined_df['psgc_10'] = combined_df.get('psgc_10_digit')
combined_df['status'] = np.where(combined_df['is_valid'], 'verified', 'invalid')
for c in combined_cols:
    if c not in combined_df.columns:
        combined_df[c] = None
combined_df = combined_df[combined_cols].reset_index(drop=True)


def write_xlsx(df: pd.DataFrame, path: str, sheet_name: str,
               tab_color: str, font_color: str = '#000000'):
    with pd.ExcelWriter(path, engine='xlsxwriter') as writer:
        df.to_excel(writer, index=False, sheet_name=sheet_name)
        wb = writer.book
        ws = writer.sheets[sheet_name]
        ws.set_tab_color(tab_color)
        hdr_fmt = wb.add_format({
            'bold': True, 'bg_color': tab_color,
            'font_color': font_color,
            'border': 1, 'text_wrap': True,
            'font_name': 'Arial', 'font_size': 10,
        })
        for i, col in enumerate(df.columns):
            ws.write(0, i, col, hdr_fmt)
            if df.empty:
                max_w = len(col) + 4
            else:
                col_max = df[col].dropna().astype(str).str.len().max()
                if pd.isna(col_max):
                    col_max = len(col)
                max_w = max(int(col_max), len(col)) + 4
            ws.set_column(i, i, min(int(max_w), 42))
        ws.freeze_panes(1, 0)
    log.info(f'Wrote {len(df):,} rows -> {path}')


write_xlsx(verified_df, VERIFIED_FILE, 'Verified', '#00B050')
write_xlsx(invalid_df, INVALID_FILE, 'Invalid', '#FF0000', font_color='#FFFFFF')
write_xlsx(combined_df, COMBINED_FILE, 'Combined', '#1F4E78', font_color='#FFFFFF')

elapsed = time.time() - t_start
print(f'\nTotal elapsed: {elapsed:.1f}s  ({elapsed/60:.2f} min)')
print('Output files:')
print(f'  ✅  {VERIFIED_FILE}  ({len(verified_df):,} rows)')
print(f'  ⚠️   {INVALID_FILE}   ({len(invalid_df):,} rows)')
print(f'  📄  {COMBINED_FILE}  ({len(combined_df):,} rows)')


13:12:01  INFO      Wrote 658 rows -> ..\..\data\output\verified\verified_addresses_parts_051_060.xlsx
13:12:03  INFO      Wrote 9,342 rows -> ..\..\data\output\invalid\invalid_addresses_parts_051_060.xlsx
13:12:06  INFO      Wrote 10,000 rows -> ..\..\data\output\combined_addresses_parts_051_060.xlsx



Total elapsed: 2634.0s  (43.90 min)
Output files:
  ✅  ..\..\data\output\verified\verified_addresses_parts_051_060.xlsx  (658 rows)
  ⚠️   ..\..\data\output\invalid\invalid_addresses_parts_051_060.xlsx   (9,342 rows)
  📄  ..\..\data\output\combined_addresses_parts_051_060.xlsx  (10,000 rows)


## Cell 11 — Results summary

Quick inline preview of both output tables for a final sanity check.

In [ ]:
print('═' * 80)
print('  PIPELINE SUMMARY')
print('═' * 80)
total = len(out_df)
n_v   = len(verified_df)
n_i   = len(invalid_df)
print(f'  Input rows        : {total:>8,}')
print(f'  Verified          : {n_v:>8,}  ({100*n_v/total:.1f}%)')
print(f'  Invalid           : {n_i:>8,}  ({100*n_i/total:.1f}%)')
print()
print('  Confidence breakdown:')
for conf, cnt in out_df['confidence'].value_counts().items():
    print(f'    {conf:<20} {cnt:>6,}  ({100*cnt/total:.1f}%)')
print()
high_c = out_df[out_df['confidence'] == 'high']
if len(high_c):
    print(f'  Avg city score (high-conf): {high_c["city_score"].mean():.1f}')
print(f'  Total elapsed     : {elapsed:.1f}s')
print('═' * 80)
print()
print('── Verified (first 10 rows) ──')
display(verified_df.head(10))
print()
print('── Invalid (all rows) ──')
display(invalid_df)


════════════════════════════════════════════════════════════════════════════════
  PIPELINE SUMMARY
════════════════════════════════════════════════════════════════════════════════
  Input rows        :   10,000
  Verified          :      658  (6.6%)
  Invalid           :    9,342  (93.4%)

  Confidence breakdown:
    low                   9,380  (93.8%)
    high                    620  (6.2%)

  Avg city score (high-conf): 97.0
  Total elapsed     : 2634.0s
════════════════════════════════════════════════════════════════════════════════

── Verified (first 10 rows) ──


,Original_Address,barangay_code,barangay,city_code,city,province_code,province,region_code,region_name
0,BLDG C-19 RAWIS COMPOUND TONDO,NaN,NaN,1.0,TONDO I/II,39.0,METRO MANILA,13.0,NATIONAL CAPITAL REGION (NCR)
1,1447 W QUINTOS ST SAMPALOC MANILA,NaN,NaN,6.0,SAMPALOC,39.0,METRO MANILA,13.0,NATIONAL CAPITAL REGION (NCR)
2,mary child hospital 667 dalupan st sampaloc ma...,NaN,NaN,6.0,SAMPALOC,39.0,METRO MANILA,13.0,NATIONAL CAPITAL REGION (NCR)
3,brg 12 cal city,NaN,NaN,1.0,CITY OF CALOOCAN,75.0,METRO MANILA,13.0,NATIONAL CAPITAL REGION (NCR)
4,1107 ALGECIRAS ST SAMPALOC MANILA,NaN,NaN,6.0,SAMPALOC,39.0,METRO MANILA,13.0,NATIONAL CAPITAL REGION (NCR)
5,6 Ligaya Pascual Street BF resort Village Talo...,17.0,TALON DOS,1.0,CITY OF LAS PI�AS,76.0,METRO MANILA,13.0,NATIONAL CAPITAL REGION (NCR)
6,3 N averilla st San Juan City,NaN,NaN,5.0,CITY OF SAN JUAN,74.0,METRO MANILA,13.0,NATIONAL CAPITAL REGION (NCR)
7,ELOISA ST SAMPALOC MANILA,NaN,NaN,6.0,SAMPALOC,39.0,METRO MANILA,13.0,NATIONAL CAPITAL REGION (NCR)
8,480A F Cayco Street Corner Alley 1 St Sampaloc...,NaN,NaN,6.0,SAMPALOC,39.0,METRO MANILA,13.0,NATIONAL CAPITAL REGION (NCR)
9,235 Salonga St Balut Tondo,NaN,NaN,1.0,TONDO I/II,39.0,METRO MANILA,13.0,NATIONAL CAPITAL REGION (NCR)



── Invalid (all rows) ──


,Original_Address,barangay_code,barangay,city_code,city,province_code,province,region_code,region_name
0,AIMATCO Builders,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Studio TV 5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Pingkian 2 Tandang Sora,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,happy homes centro brgy lag on dcn,NaN,NaN,NaN,NaN,34.0,LAGUNA,4.0,REGION IV-A (CALABARZON)
4,97 blk4 lot 26 oxford st Hampstead place,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
9337,Golden meadow,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9338,B8D L16 P2 A3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9339,"Unit 01, 9011 Congressional ave,",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9340,Mother Seton Hospital,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Cell 12 — One-shot re-run helper

Use this cell to re-run the entire pipeline in one shot after changing config in Cell 1.  
It re-uses all the functions defined above — no need to re-run Cells 3–6 unless you change the reference files.

In [ ]:
def run_pipeline(
    input_file   = INPUT_FILE,
    dim_path     = DIM_LOCATION,
    alias_path   = ALIAS_FILE,
    out_verified = OUT_VERIFIED,
    out_invalid  = OUT_INVALID,
    max_rows     = MAX_ROWS,
    use_api      = False,
    api_query_mode = API_QUERY_MODE,
):
    t0 = time.time()
    _dim, _alias, _cities, _provinces, _regions, _barangays = load_reference(
        dim_path, alias_path
    )
    _re, _rmap = build_alias_regex(_alias)

    df = pd.read_excel(input_file)
    if max_rows:
        df = df.iloc[:max_rows]
    col = df.columns[0]

    addresses = df[col].fillna('').astype(str).tolist()
    normalized = [normalize_alias(x, _re, _rmap) for x in addresses]

    recs = []
    for i, addr_norm in enumerate(tqdm(normalized, total=len(normalized), desc='Fuzzy', unit='row')):
        rec = {
            'original_address': addresses[i],
            'address_normalized': addr_norm,
            'api_status': 'skipped',
            'osm_city': None, 'osm_province': None, 'osm_region': None,
            'osm_display': None, 'osm_lat': None, 'osm_lon': None,
        }
        rec.update(match_address(
            addr_norm,
            _dim, _cities, _provinces, _regions, _barangays,
        ))
        rec['needs_api'] = False
        recs.append(rec)

    if use_api:
        log.warning('API path is disabled in this fast profile; running fuzzy-only instead')

    merged = pd.DataFrame(recs)
    if 'city_required_pass' not in merged.columns:
        merged['city_required_pass'] = False
    merged['city_required_pass'] = merged['city_required_pass'].fillna(False).astype(bool)

    # Strict rule: valid only when city evidence is present in address.
    merged['is_valid'] = merged['city_required_pass']

    base_cols = [
        'original_address',
        'barangay_code', 'barangay_name',
        'city_code', 'city_name',
        'province_code', 'province_name',
        'region_code', 'region_name',
    ]
    base_cols = [c for c in base_cols if c in merged.columns]

    v_df = merged[merged['is_valid']][base_cols].reset_index(drop=True)
    i_df = merged[~merged['is_valid']][base_cols].reset_index(drop=True)

    write_xlsx(v_df, out_verified, 'Verified', '#00B050')
    write_xlsx(i_df, out_invalid, 'Invalid', '#FF0000', font_color='#FFFFFF')

    elapsed = time.time() - t0
    log.info(f'Done in {elapsed:.1f}s  |  Verified: {len(v_df):,}  Invalid: {len(i_df):,}')
    return v_df, i_df


# Uncomment to run:
# verified_df, invalid_df = run_pipeline()
# verified_df, invalid_df = run_pipeline(max_rows=1000)


In [ ]:
# Cell 13 -- Invalid pattern diagnostics (batch profile)
diag_df = out_df.copy()
diag_df['orig'] = diag_df['original_address'].fillna('').astype(str)
diag_df['norm'] = diag_df['address_normalized'].fillna('').astype(str)
diag_df['invalid_reason'] = diag_df.get('invalid_reason', pd.Series([None] * len(diag_df))).fillna('resolved')

invalid_only = diag_df[~diag_df['is_valid']].copy()
print(f'Total rows: {len(diag_df):,}')
print(f'Invalid rows: {len(invalid_only):,}')

print('\nTop invalid reasons:')
print(invalid_only['invalid_reason'].value_counts().head(10).to_string())

# Lightweight pattern flags for error clustering
pat = pd.DataFrame(index=invalid_only.index)
pat['has_brgy_word'] = invalid_only['norm'].str.contains(r'\b(?:BRGY|BARANGAY|BGY)\b', regex=True, na=False)
pat['has_city_word'] = invalid_only['norm'].str.contains(r'\b(?:CITY|MUNICIPALITY|MUN)\b', regex=True, na=False)
pat['has_numbers'] = invalid_only['norm'].str.contains(r'\d', regex=True, na=False)
pat['short_addr'] = invalid_only['norm'].str.len() <= 28
pat['street_only_tokens'] = invalid_only['norm'].str.contains(
    r'\b(?:ST|STREET|AVE|AVENUE|ROAD|RD|BLK|LOT|UNIT|PHASE|B\d+|L\d+)\b', regex=True, na=False
)

cluster = pd.DataFrame({
    'missing_city_word': (~pat['has_city_word']).astype(int),
    'with_brgy_word': pat['has_brgy_word'].astype(int),
    'street_only_tokens': pat['street_only_tokens'].astype(int),
    'short_addr': pat['short_addr'].astype(int),
})
print('\nPattern prevalence among invalid rows:')
print(cluster.mean().mul(100).round(1).astype(str) + '%')

print('\nSample invalid addresses (first 30):')
display(invalid_only[['original_address', 'address_normalized', 'city_signal', 'city_name', 'province_name', 'invalid_reason']].head(30))

# Show likely false-invalid candidates (have city-like cues but still invalid)
candidates = invalid_only[
    invalid_only['norm'].str.contains(
        r'\b(?:PQUE|PARANAQUE|PARAÑAQUE|CEBU|QC|QUEZON|MANILA|MAKATI|TAGUIG)\b',
        regex=True, na=False
    )
]
print(f'\nPotentially recoverable invalids with city-like cues: {len(candidates):,}')
display(candidates[['original_address', 'address_normalized', 'city_signal', 'city_name', 'province_name', 'invalid_reason']].head(30))

Total rows: 10,000
Invalid rows: 9,342

Top invalid reasons:
invalid_reason
missing_explicit_city    9342

Pattern prevalence among invalid rows:
missing_city_word     96.9%
with_brgy_word         7.2%
street_only_tokens    55.6%
short_addr            61.3%
dtype: str

Sample invalid addresses (first 30):


,original_address,address_normalized,city_signal,city_name,province_name,invalid_reason
620,AIMATCO Builders,AIMATCO BUILDERS,NaN,NaN,NaN,missing_explicit_city
621,Studio TV 5,STUDIO TV 5,NaN,NaN,NaN,missing_explicit_city
622,Pingkian 2 Tandang Sora,PINGKIAN 2 TANDANG SORA,NaN,NaN,NaN,missing_explicit_city
623,happy homes centro brgy lag on dcn,HAPPY HOMES CENTRO BARANGAY LAG ON DCN,NaN,NaN,LAGUNA,missing_explicit_city
624,97 blk4 lot 26 oxford st Hampstead place,97 BLK4 LOT 26 OXFORD STREET HAMPSTEAD PLACE,NaN,NaN,NaN,missing_explicit_city
625,116 Elon Elon St East Bagong barrio,116 ELON ELON STREET EAST BAGONG BARANGAY,NaN,NaN,EASTERN SAMAR,missing_explicit_city
626,BLK 6 LOT 21 GUILMAR PLACE SUBD,BLOCK 6 LOT 21 GUILMAR PLACE SUBDIVISION,NaN,NaN,NaN,missing_explicit_city
627,Lancaster Residence,LANCASTER RESIDENCE,NaN,NaN,NaN,missing_explicit_city
628,unit 7 9273 tade apartment,UNIT 7 9273 TADE APARTMENT,NaN,NaN,NaN,missing_explicit_city
629,51 JOVAN ST VERAVILLE TOWNHOMES 5A TALON 5,51 JOVAN STREET VERAVILLE TOWNHOMES 5A TALON 5,NaN,NaN,NaN,missing_explicit_city



Potentially recoverable invalids with city-like cues: 96


,original_address,address_normalized,city_signal,city_name,province_name,invalid_reason
634,Residencial de Manila,RESIDENCIAL DE MANILA,NaN,NaN,METRO MANILA,missing_explicit_city
847,123 carlos palanca st qpo manila,123 CARLOS PALANCA STREET QPO MANILA,NaN,NaN,METRO MANILA,missing_explicit_city
915,2420-B ELIAS STREET STACRUZ MANILA,2420 B ELIAS STREET STACRUZ MANILA,NaN,NaN,METRO MANILA,missing_explicit_city
971,TOWER 2 UNIT 15JK CELADON PARK MANILA,TOWER 2 UNIT 15JK CELADON PARK MANILA,NaN,NaN,METRO MANILA,missing_explicit_city
1082,411 T Earnshow manila,411 T EARNSHOW MANILA,NaN,NaN,METRO MANILA,missing_explicit_city
1152,escolta manila,ESCOLTA MANILA,NaN,NaN,METRO MANILA,missing_explicit_city
1169,220 D Ampil St Sta Mesa Manila brgy 631,220 D AMPIL STREET SANTA MESA MANILA BARANGAY 631,NaN,NaN,METRO MANILA,missing_explicit_city
1228,Singalong manila,SINGALONG MANILA,NaN,NaN,METRO MANILA,missing_explicit_city
1233,"TWO HEARTS BOARDING HOME ,GATE IN BETWEEN UNIV...",TWO HEARTS BOARDING HOME GATE IN BETWEEN UNIVE...,NaN,NaN,NORTHERN SAMAR,missing_explicit_city
1258,Torre de Manila,TORRE DE MANILA,NaN,NaN,METRO MANILA,missing_explicit_city


In [ ]:
# Cell 14 -- Recoverable invalids snapshot (plain text)
cand = out_df[
    (~out_df['is_valid']) &
    out_df['address_normalized'].fillna('').astype(str).str.contains(
        r'\b(?:PQUE|PARANAQUE|PARAÑAQUE|QC|QUEZON|MANILA|MKT|MAKATI|TAGUIG|PASIG|MARIKINA|CEBU|DAVAO|CDO|CAGAYAN DE ORO)\b',
        regex=True, na=False
    )
].copy()

print(f'Recoverable candidates: {len(cand):,}')
cols = ['original_address', 'address_normalized', 'city_signal', 'city_name', 'province_name', 'invalid_reason']
for i, row in cand[cols].head(60).iterrows():
    print('-' * 100)
    print(f"ORIG : {row['original_address']}")
    print(f"NORM : {row['address_normalized']}")
    print(f"SIG  : {row['city_signal']} | CITY: {row['city_name']} | PROV: {row['province_name']} | REASON: {row['invalid_reason']}")

print('\nCity-not-resolved rows:')
unres = out_df[out_df['invalid_reason'] == 'city_not_resolved'][cols]
for i, row in unres.head(60).iterrows():
    print('-' * 100)
    print(f"ORIG : {row['original_address']}")
    print(f"NORM : {row['address_normalized']}")
    print(f"SIG  : {row['city_signal']} | CITY: {row['city_name']} | PROV: {row['province_name']} | REASON: {row['invalid_reason']}")

Recoverable candidates: 96
----------------------------------------------------------------------------------------------------
ORIG : Residencial de Manila
NORM : RESIDENCIAL DE MANILA
SIG  : nan | CITY: nan | PROV: METRO MANILA | REASON: missing_explicit_city
----------------------------------------------------------------------------------------------------
ORIG : 123 carlos palanca st qpo manila
NORM : 123 CARLOS PALANCA STREET QPO MANILA
SIG  : nan | CITY: nan | PROV: METRO MANILA | REASON: missing_explicit_city
----------------------------------------------------------------------------------------------------
ORIG : 2420-B ELIAS STREET STACRUZ MANILA
NORM : 2420 B ELIAS STREET STACRUZ MANILA
SIG  : nan | CITY: nan | PROV: METRO MANILA | REASON: missing_explicit_city
----------------------------------------------------------------------------------------------------
ORIG : TOWER 2 UNIT 15JK CELADON PARK MANILA
NORM : TOWER 2 UNIT 15JK CELADON PARK MANILA
SIG  : nan | CITY: nan | 

----------------------------------------------------------------------------------------------------
ORIG : 416 Verdad st sampaloc
NORM : 416 VERDAD STREET SAMPALOC
SIG  : SAMPALOC | CITY: nan | PROV: nan | REASON: city_not_resolved
----------------------------------------------------------------------------------------------------
ORIG : 1243 C NAVARRA ST SAMPALOC
NORM : 1243 C NAVARRA STREET SAMPALOC
SIG  : SAMPALOC | CITY: nan | PROV: nan | REASON: city_not_resolved
----------------------------------------------------------------------------------------------------
ORIG : UNIT T 1441 MACEDA ST SAMPALOC
NORM : UNIT T 1441 MACEDA STREET SAMPALOC
SIG  : SAMPALOC | CITY: nan | PROV: nan | REASON: city_not_resolved
----------------------------------------------------------------------------------------------------
ORIG : Blk 4 Lot 3 San carlos villa 1
NORM : BLOCK 4 LOT 3 SAN CARLOS CITY VILLA 1
SIG  : CITY OF SAN CARLOS | CITY: nan | PROV: AGUSAN DEL NORTE | REASON: city_not_resolved
--

In [ ]:
# Cell 15 -- Compact unresolved diagnostics
tmp = out_df.copy()
tmp['norm'] = tmp['address_normalized'].fillna('').astype(str)

unres = tmp[tmp['invalid_reason'] == 'city_not_resolved'].copy()
print(f"city_not_resolved rows: {len(unres):,}")
for s in unres['norm'].head(20).tolist():
    print(' -', s[:140])

# Count metro-manila style abbreviations still not resolved
abbr_patterns = {
    'PQUE': r'\bPQUE\b',
    'MKT': r'\bMKT\b',
    'QC': r'\bQC\b',
    'MNL': r'\bMNL\b',
    'BGC': r'\bBGC\b',
    'CDO': r'\bCDO\b',
}
print('\nUnresolved abbreviation counts:')
for k, pat in abbr_patterns.items():
    c = int((~tmp['is_valid'] & tmp['norm'].str.contains(pat, regex=True, na=False)).sum())
    print(f' {k}: {c}')

city_not_resolved rows: 38
 - SAN CARLOS CITY HEIGHTS
 - ZONE 3 SAN CARLOS CITY STREET
 - 731 DON QUIJOTE STREET SAMPALOC MLA
 - 1984 LOYOLA STREET SAMPALOC
 - 9TH STREET SAMPALOC MANSANITAS CORNER
 - 1120 B G TOLENTINO STREET SAMPALOC
 - 23 JACOB PUTOL STREET NAGA CITY
 - 1173 CAROLA STREET SAMPALOC
 - 828 MACEDA STREET SAMPALOC
 - 851 ALEX STREET SAMPALOC
 - 1230 SAN DIEGO STREET SAMPALOC MANILA
 - 351 CLAVEL STREET SAN NICOLAS MANILA
 - SAN CARLOS CITY HEIGHTS
 - 1115 MIGUELIN STREET SAMPALOC
 - 605 C GERONIMO STREET SAMPALOC
 - 1032 B BITINA STREET SAMPALOC
 - 516 GERONIMO STREET SAMPALOC
 - 416 VERDAD STREET SAMPALOC
 - 1243 C NAVARRA STREET SAMPALOC
 - UNIT T 1441 MACEDA STREET SAMPALOC

Unresolved abbreviation counts:
 PQUE: 0
 MKT: 0
 QC: 0
 MNL: 0
 BGC: 0
 CDO: 0


In [ ]:
# Cell 16 -- Unresolved detail view (plain text sample)
cols = ['original_address', 'address_normalized', 'city_signal', 'province_name', 'invalid_reason']
u = out_df[out_df['invalid_reason'] == 'city_not_resolved'][cols].copy()
print(f'city_not_resolved count: {len(u):,}')
for i, row in u.head(25).iterrows():
    print('-' * 90)
    print('ORIG :', str(row['original_address'])[:150])
    print('NORM :', str(row['address_normalized'])[:150])
    print('SIG  :', row['city_signal'], '| PROV:', row['province_name'], '| REASON:', row['invalid_reason'])

city_not_resolved count: 38
------------------------------------------------------------------------------------------
ORIG : San Carlos Heights
NORM : SAN CARLOS CITY HEIGHTS
SIG  : CITY OF SAN CARLOS | PROV: AGUSAN DEL NORTE | REASON: city_not_resolved
------------------------------------------------------------------------------------------
ORIG : Zone 3 San Carlos Street
NORM : ZONE 3 SAN CARLOS CITY STREET
SIG  : CITY OF SAN CARLOS | PROV: AGUSAN DEL NORTE | REASON: city_not_resolved
------------------------------------------------------------------------------------------
ORIG : 731 Don Quijote St Sampaloc Mla
NORM : 731 DON QUIJOTE STREET SAMPALOC MLA
SIG  : SAMPALOC | PROV: nan | REASON: city_not_resolved
------------------------------------------------------------------------------------------
ORIG : 1984 LOYOLA ST SAMPALOC
NORM : 1984 LOYOLA STREET SAMPALOC
SIG  : SAMPALOC | PROV: nan | REASON: city_not_resolved
----------------------------------------------------------------

In [ ]:
# Cell 17 -- Focus checks (La Carlota and Tondo)
tmp = out_df.copy()
tmp['norm'] = tmp['address_normalized'].fillna('').astype(str)

la = tmp[tmp['city_name'].fillna('').str.contains('LA CARLOTA', case=False, na=False)].copy()
la_false = la[~la['norm'].str.contains('LA CARLOTA', case=False, na=False)]
print(f'Rows with city_name=LA CARLOTA*: {len(la):,}')
print(f'LA CARLOTA assigned without mention: {len(la_false):,}')

tondo_rows = tmp[tmp['norm'].str.contains(r'\bTONDO\b', case=False, regex=True, na=False)].copy()
print(f'Rows mentioning TONDO: {len(tondo_rows):,}')
if len(tondo_rows):
    print('TONDO status counts:')
    print(tondo_rows['is_valid'].value_counts().to_string())

Rows with city_name=LA CARLOTA*: 0
LA CARLOTA assigned without mention: 0
Rows mentioning TONDO: 210
TONDO status counts:
is_valid
True     208
False      2


In [ ]:
# Cell 18 -- Manila district city-key check from dim_location city_name
manila_cores = [
    'TONDO', 'BINONDO', 'ERMITA', 'INTRAMUROS', 'MALATE', 'PACO',
    'PANDACAN', 'PORT AREA', 'QUIAPO', 'SAMPALOC', 'SAN NICOLAS',
    'SANTA ANA', 'SANTA CRUZ',
]

def _canon_tmp(s: str) -> str:
    return re.sub(r'\s+', ' ', re.sub(r'[^A-Z0-9]+', ' ', str(s).upper())).strip()

canon_cities = [_canon_tmp(c) for c in cities]
print('Has CITY OF MANILA:', 'CITY OF MANILA' in canon_cities)
print('Has MANILA:', 'MANILA' in canon_cities)

print('\nDistrict core coverage in city_name:')
for core in manila_cores:
    ccore = _canon_tmp(core)
    hits = sorted({cities[i] for i, c in enumerate(canon_cities) if f' {ccore} ' in f' {c} '})
    print(f'[{core}] -> {len(hits)} match(es)')
    for h in hits[:8]:
        print('  -', h)

Has CITY OF MANILA: False
Has MANILA: False

District core coverage in city_name:
[TONDO] -> 1 match(es)
  - TONDO I/II
[BINONDO] -> 1 match(es)
  - BINONDO
[ERMITA] -> 1 match(es)
  - ERMITA
[INTRAMUROS] -> 1 match(es)
  - INTRAMUROS
[MALATE] -> 1 match(es)
  - MALATE
[PACO] -> 1 match(es)
  - PACO
[PANDACAN] -> 1 match(es)
  - PANDACAN
[PORT AREA] -> 1 match(es)
  - PORT AREA
[QUIAPO] -> 1 match(es)
  - QUIAPO
[SAMPALOC] -> 1 match(es)
  - SAMPALOC
[SAN NICOLAS] -> 1 match(es)
  - SAN NICOLAS
[SANTA ANA] -> 1 match(es)
  - SANTA ANA
[SANTA CRUZ] -> 1 match(es)
  - SANTA CRUZ


In [ ]:
# Cell 19 -- Tondo mapping probe in dim_location
probe = dim[dim['barangay_name'].fillna('').astype(str).str.contains('TONDO', case=False, na=False)].copy()
print(f'Rows with barangay containing TONDO: {len(probe):,}')
if len(probe):
    print('Unique city names:')
    for c in sorted(probe['city_name'].dropna().unique().tolist())[:20]:
        print(' -', c)
    print('Unique province names:')
    for p in sorted(probe['province_name'].dropna().unique().tolist())[:10]:
        print(' -', p)

Rows with barangay containing TONDO: 6
Unique city names:
 - ANDA
 - CABANGAN
 - CLAVER
 - SAN JOSE CITY
 - SAN JUAN
 - TANGALAN
Unique province names:
 - AKLAN
 - LA UNION
 - NUEVA ECIJA
 - PANGASINAN
 - SURIGAO DEL NORTE
 - ZAMBALES


In [ ]:
# Targeted Manila district sanity check
for raw in ['TONDO MANILA', 'BINONDO MANILA', 'ERMITA MANILA', 'INTRAMUROS MANILA']:
    addr = normalize_alias(raw, compiled_re, replace_map)
    ans = match_address(addr, dim, cities, provinces, regions, barangays)
    print(raw, '=>', ans['city_signal'], '|', ans['city_name'], '|', ans['invalid_reason'])

TONDO MANILA => TONDO I/II | TONDO I/II | None
BINONDO MANILA => BINONDO | BINONDO | None
ERMITA MANILA => ERMITA | ERMITA | None
INTRAMUROS MANILA => INTRAMUROS | INTRAMUROS | None
